# OAC Catalog ACL Transformer — Silver Layer

**Version:** 2.0 — Standard Catalog source, two-stage write, admin detection, OCI GenAI enrichments  
**Source:** `cbtest_standard_catalog.default.OAC_CATALOG_ACL_BRONZE`  
**Stage 1 target:** `cbtest_standard_catalog.default.OAC_CATALOG_ACL_SILVER` (Delta)  
**Stage 2 target:** `arganoadw_oacuser_sh.oacuser.OAC_CATALOG_ACL_SILVER` (ADW)

---
### Enrichments Applied

**PySpark (v1 — preserved):**
1. `ITEM_PATH_DECODED` — Base64 decode `ITEM_ID` to readable catalog path
2. `CATALOG_TYPE_LABEL` — proper case (`workbooks` → `Workbooks`)
3. `ITEM_MODIFIED_TS` — ISO string → Spark `TimestampType`
4. `PERMISSION_LEVEL` — 6 boolean flags → single readable label
5. `RISK_SCORE` — weighted integer 0–10
6. `RISK_LEVEL` — score bucket label (Critical / High / Medium / Low / None)
7. `ACCOUNT_CATEGORY` — clean principal type label
8. `EXTRACTED_AT_TS` — ISO string → `TimestampType`
9. `HAS_CREATED_DATE` — null quality flag (0/1)
10. `DATA_QUALITY_FLAG` — row-level completeness indicator

**GenAI (v2 — new):**
11. `IS_ADMIN_ACCOUNT` — admin flag (name pattern + Full Control breadth detection)
12. `RISK_NARRATIVE` — plain-English risk summary per account via `google.gemini-2.5-pro` (Agent B). Admin accounts receive a standard override note — not a risk flag.
13. `INFERRED_OWNER` — suggested owner for orphaned/service-account-owned objects via GenAI (Agent C)
14. `INFERRED_OWNER_NOTE` — one-sentence rationale from the model

---
### Pre-requisites
1. Bronze notebook completed successfully — Delta table exists at Stage 1 source
2. Spark cluster attached to this notebook
3. `OCI_GENAI_COMPARTMENT_ID` set in Section 1 before running Sections 7–9

## Section 1 — Imports & Configuration
All configuration defined here — source/target tables, risk weights, admin patterns, and GenAI settings.

**Risk score weight design:**
- `WEIGHT_CHANGE_PERM` (4) — highest risk: can modify other users' access
- `WEIGHT_TAKE_OWN` (3) — can reassign ownership, bypassing access controls
- `WEIGHT_DELETE` (2) — destructive: can permanently remove the item
- `WEIGHT_WRITE` (1) — data modification risk

**Admin detection:**
- `ADMIN_NAME_PATTERNS` — case-insensitive substring match against `ACCOUNT_NAME`. Matching accounts receive `ADMIN_NARRATIVE` and skip GenAI.
- `ADMIN_FC_THRESHOLD` — accounts with Full Control on this many distinct catalog types are flagged as admin regardless of name.

**GenAI model:** `google.gemini-2.5-pro` — available in this AIDP instance. Update `OCI_GENAI_MODEL_ID` to switch models.  
> **Action required:** Set `OCI_GENAI_COMPARTMENT_ID` before running Sections 7–9.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, TimestampType
import base64
import json
import time
from datetime import datetime, timezone
from collections import defaultdict

# ── Source & Target ──────────────────────────────────────────
BRONZE_FULL   = 'cbtest_standard_catalog.default.OAC_CATALOG_ACL_BRONZE'
SILVER_STAGE1 = 'cbtest_standard_catalog.default.OAC_CATALOG_ACL_SILVER'
SILVER_STAGE2 = 'arganoadw_oacuser_sh.oacuser.OAC_CATALOG_ACL_SILVER'

# ── Risk Score Weights ────────────────────────────────────────
WEIGHT_CHANGE_PERM  = 4
WEIGHT_TAKE_OWN     = 3
WEIGHT_DELETE       = 2
WEIGHT_WRITE        = 1

# ── Admin Account Detection ───────────────────────────────────
ADMIN_NAME_PATTERNS = [
    'admin', 'administrator', 'bi_administrator',
    'biserviceadministrator', 'sysadmin', 'oac-admin', 'catalog-admin',
]
ADMIN_FC_THRESHOLD = 3
ADMIN_NARRATIVE = (
    'This is an administrative or privileged account. '
    'Elevated access across catalog types is expected and by design. '
    'No remediation required. Recommend periodic verification that '
    'this account is actively managed and assigned to the correct owner.'
)

# ── Owner Inference Triggers ──────────────────────────────────
SERVICE_ACCOUNT_PATTERNS = ['svc-', 'service-', 'shared-', 'system-', 'oac-svc', 'oacservice']

# ── OCI GenAI Configuration ───────────────────────────────────
# ⚠️  ACTION REQUIRED: set your compartment OCID below
OCI_GENAI_MODEL_ID       = 'google.gemini-2.5-pro'
OCI_GENAI_ENDPOINT       = 'https://inference.generativeai.us-chicago-1.oci.oraclecloud.com'
OCI_GENAI_COMPARTMENT_ID = 'ocid1.compartment.oc1..aaaaaaaa7g7ilirhktswqmci2n5tgghpadrs7qu5xnoo6mzaoh23z6eofwva'
GENAI_MAX_TOKENS         = 350
GENAI_TEMPERATURE        = 0.1
GENAI_RATE_LIMIT_SLEEP   = 0.5

print('=' * 55)
print('  SECTION 1 COMPLETE: Imports & Configuration')
print(f'  Source  : {BRONZE_FULL}')
print(f'  Stage 1 : {SILVER_STAGE1}')
print(f'  Stage 2 : {SILVER_STAGE2}')
print(f'  Model   : {OCI_GENAI_MODEL_ID}')
print(f'  Risk weights: ChangePerm={WEIGHT_CHANGE_PERM}, TakeOwn={WEIGHT_TAKE_OWN}, Delete={WEIGHT_DELETE}, Write={WEIGHT_WRITE}')
print('=' * 55)

## Section 2 — Spark Session & Read Bronze
Reads from the Standard Catalog Bronze Delta table written by the Bronze notebook. No wallet, no JDBC — Spark resolves the 3-part path via the Standard Catalog metadata layer.

The Bronze table is the raw, unmodified API extract and is the source of truth. It is **never written to from this notebook**. All enrichment goes to the Silver table only.

**v2 change:** source is now the Standard Catalog Delta table rather than the ADW External Catalog. This decouples the Silver read from the ADW connection — Silver can run in any AIDP instance with the Bronze Delta table, no ADW connection required for the read path.

In [ ]:
spark = SparkSession.builder.appName('oac_acl_silver').getOrCreate()

df_bronze    = spark.table(BRONZE_FULL)
total_bronze = df_bronze.count()

print('=' * 55)
print('  SECTION 2 COMPLETE: Bronze Table Loaded')
print(f'  Source    : {BRONZE_FULL}')
print(f'  Rows read : {total_bronze:,}')
print(f'  Columns   : {len(df_bronze.columns)}')
print('=' * 55)

## Section 3 — UDFs
Four Python functions registered as Spark UDFs for row-level transformation logic.

- **`decode_base64_id`** — reverses the Base64 URL-safe encoding applied by the Bronze extraction. e.g. `L3NoYXJlZC9TYWxlcw` → `/shared/Sales`. Returns original value if decode fails — no silent data loss.
- **`get_permission_level`** — maps 6 boolean flags to a single label (highest privilege wins): `Full Control` → `Read-Write-Delete` → `Read-Write` → `Read-Only` → `No Access`
- **`get_risk_score`** — weighted integer 0–10 using `WEIGHT_*` constants from Section 1. Re-run this section if weights are changed.
- **`get_account_category`** — maps `ACCOUNT_TYPE` (`ApplicationRole` / `User`) to `Application Role` / `Individual User`. Roles like `BIServiceAdministrator` are not individual people — this UDF makes the distinction visible in the OAC report.

In [ ]:
def decode_base64_id(encoded_id):
    if not encoded_id:
        return None
    try:
        padded = encoded_id + '=' * (4 - len(encoded_id) % 4)
        return base64.urlsafe_b64decode(padded).decode('utf-8')
    except Exception:
        return encoded_id


def get_permission_level(read, write, delete, change_perm, take_own):
    if change_perm == 1 or take_own == 1:
        return 'Full Control'
    elif delete == 1:
        return 'Read-Write-Delete'
    elif write == 1:
        return 'Read-Write'
    elif read == 1:
        return 'Read-Only'
    else:
        return 'No Access'


def get_risk_score(write, delete, change_perm, take_own):
    if write is None and delete is None and change_perm is None and take_own is None:
        return 0
    score = 0
    if change_perm == 1: score += WEIGHT_CHANGE_PERM
    if take_own    == 1: score += WEIGHT_TAKE_OWN
    if delete      == 1: score += WEIGHT_DELETE
    if write       == 1: score += WEIGHT_WRITE
    return score


def get_account_category(account_type, account_guid):
    if account_type == 'ApplicationRole':
        return 'Application Role'
    elif account_type == 'User':
        return 'Individual User'
    elif account_guid and '@' in account_guid:
        return 'Individual User'
    else:
        return 'Unknown'


udf_decode_id        = F.udf(decode_base64_id,    StringType())
udf_permission_level = F.udf(get_permission_level, StringType())
udf_risk_score       = F.udf(get_risk_score,       IntegerType())
udf_account_category = F.udf(get_account_category, StringType())

print('=' * 55)
print('  SECTION 3 COMPLETE: UDFs Registered')
print('  UDFs: decode_base64_id, get_permission_level,')
print('        get_risk_score, get_account_category')
print('=' * 55)

## Section 4 — PySpark Transformations (10 enrichments)
All 10 enrichments applied in a chained `withColumn()` pattern. Bronze originals first in the final `.select()`, Silver enrichments grouped at the end for clear lineage.

| # | Column | Description |
|---|--------|-------------|
| 1 | `ITEM_PATH_DECODED` | Base64 → readable catalog path |
| 2 | `CATALOG_TYPE_LABEL` | Proper case — `workbooks` → `Workbooks` |
| 3 | `ITEM_MODIFIED_TS` | ISO string → `TimestampType` |
| 4 | `PERMISSION_LEVEL` | 6 flags → readable label |
| 5 | `RISK_SCORE` | Weighted integer 0–10 |
| 6 | `RISK_LEVEL` | Critical / High / Medium / Low / None |
| 7 | `ACCOUNT_CATEGORY` | Application Role / Individual User |
| 8 | `EXTRACTED_AT_TS` | ISO string → `TimestampType` |
| 9 | `HAS_CREATED_DATE` | 0/1 quality flag — `ITEM_CREATED` blank on this OAC instance |
| 10 | `DATA_QUALITY_FLAG` | `MISSING_PRINCIPAL` / `MISSING_PERMISSIONS` / `OK` |

> **`ITEM_CREATED` hooks A–D** are commented out — activate when the field is populated on the target OAC instance.

In [ ]:
df_silver = (
    df_bronze

    # 1. Decode Base64-encoded ITEM_ID to readable catalog path
    .withColumn('ITEM_PATH_DECODED', udf_decode_id(F.col('ITEM_ID')))

    # 2. CATALOG_TYPE_LABEL — proper case
    .withColumn('CATALOG_TYPE_LABEL', F.initcap(F.col('CATALOG_TYPE')))

    # 3. Cast ITEM_MODIFIED to proper timestamp
    .withColumn('ITEM_MODIFIED_TS', F.to_timestamp(F.col('ITEM_MODIFIED')))

    # 4. PERMISSION_LEVEL — map 6 flags to readable label
    .withColumn('PERMISSION_LEVEL',
        udf_permission_level(F.col('PERM_READ'), F.col('PERM_WRITE'),
                             F.col('PERM_DELETE'), F.col('PERM_CHANGE_PERM'), F.col('PERM_TAKE_OWN')))

    # 5. RISK_SCORE — weighted integer 0–10
    .withColumn('RISK_SCORE',
        udf_risk_score(F.col('PERM_WRITE'), F.col('PERM_DELETE'),
                       F.col('PERM_CHANGE_PERM'), F.col('PERM_TAKE_OWN')))

    # 6. RISK_LEVEL — bucket RISK_SCORE
    .withColumn('RISK_LEVEL',
        F.when(F.col('RISK_SCORE') >= 8, 'Critical')
         .when(F.col('RISK_SCORE') >= 5, 'High')
         .when(F.col('RISK_SCORE') >= 2, 'Medium')
         .when(F.col('RISK_SCORE') == 0, 'None')
         .otherwise('Low'))

    # 7. ACCOUNT_CATEGORY — clean principal type
    .withColumn('ACCOUNT_CATEGORY',
        udf_account_category(F.col('ACCOUNT_TYPE'), F.col('ACCOUNT_GUID')))

    # 8. Cast EXTRACTED_AT to proper timestamp
    .withColumn('EXTRACTED_AT_TS', F.to_timestamp(F.col('EXTRACTED_AT')))

    # 9. HAS_CREATED_DATE — flag rows missing creation date
    .withColumn('HAS_CREATED_DATE',
        F.when(F.col('ITEM_CREATED').isNull() | (F.col('ITEM_CREATED') == ''), F.lit(0)).otherwise(F.lit(1)))

    # ── ITEM_CREATED HOOKS (activate when field is populated) ──
    # Hook A: .withColumn('ITEM_CREATED_TS', F.to_timestamp(F.col('ITEM_CREATED')))
    # Hook B: .withColumn('ITEM_AGE_DAYS',   F.datediff(F.col('ITEM_MODIFIED_TS'), F.col('ITEM_CREATED_TS')))
    # Hook C: .withColumn('ITEM_AGE_BUCKET', F.when(F.col('ITEM_AGE_DAYS') <= 30, '< 30 days')
    #                                          .when(F.col('ITEM_AGE_DAYS') <= 90, '30-90 days')
    #                                          .when(F.col('ITEM_AGE_DAYS') <= 365, '90-365 days').otherwise('> 1 year'))
    # Hook D: add ITEM_CREATED, ITEM_CREATED_TS, ITEM_AGE_DAYS, ITEM_AGE_BUCKET to .select() below
    # ──────────────────────────────────────────────────────────

    # 10. DATA_QUALITY_FLAG — row-level completeness
    .withColumn('DATA_QUALITY_FLAG',
        F.when(F.col('ACCOUNT_GUID').isNull() | F.col('ACCOUNT_NAME').isNull(), 'MISSING_PRINCIPAL')
         .when(F.col('PERM_READ').isNull() & F.col('PERM_WRITE').isNull(), 'MISSING_PERMISSIONS')
         .otherwise('OK'))

    .select(
        # Bronze originals
        F.col('CATALOG_TYPE'), F.col('ITEM_ID'), F.col('ITEM_NAME'),
        F.col('ITEM_PATH').alias('ITEM_PATH_RAW'), F.col('ITEM_OWNER'),
        F.col('ITEM_MODIFIED'), F.col('ITEM_CREATED'),
        F.col('ACCOUNT_GUID'), F.col('ACCOUNT_TYPE'), F.col('ACCOUNT_NAME'),
        F.col('PERM_READ'), F.col('PERM_WRITE'), F.col('PERM_LIST'),
        F.col('PERM_DELETE'), F.col('PERM_CHANGE_PERM'), F.col('PERM_TAKE_OWN'),
        F.col('EXTRACTED_AT'),
        # Silver enrichments
        F.col('CATALOG_TYPE_LABEL'), F.col('ITEM_PATH_DECODED').alias('ITEM_PATH'),
        F.col('ITEM_MODIFIED_TS'), F.col('ACCOUNT_CATEGORY'),
        F.col('PERMISSION_LEVEL'), F.col('RISK_SCORE'), F.col('RISK_LEVEL'),
        F.col('HAS_CREATED_DATE'), F.col('DATA_QUALITY_FLAG'), F.col('EXTRACTED_AT_TS')
    )
)

total_silver = df_silver.count()
ok_rows      = df_silver.filter(F.col('DATA_QUALITY_FLAG') == 'OK').count()

print('=' * 55)
print('  SECTION 4 COMPLETE: Transformations Applied')
print(f'  Total rows   : {total_silver:,}')
print(f'  Quality OK   : {ok_rows:,}')
print(f'  Flagged rows : {total_silver - ok_rows:,}')
print('=' * 55)

## Section 5 — Quality Summary
Three distribution tables printed **before writing** — validate that enrichment results look correct before committing data.

If distributions look wrong (e.g. all rows showing `No Access`, unexpected `RISK_LEVEL` values), stop here, investigate the Bronze data, and re-run Section 4.

**Expected distributions based on current data:**
- `PERMISSION_LEVEL`: majority `Read-Only` or `Full Control`
- `RISK_LEVEL`: mix of `High` and `None` (two `ApplicationRole` entries)
- `ACCOUNT_CATEGORY`: `Application Role` dominant — expected, the API returns role-based ACLs primarily

> **Note:** `df.show()` is retained here intentionally — this is a pre-write validation gate running on aggregated distributions (small row count), not on the full Silver DataFrame. Safe at any scale.

In [ ]:
print('\n📊 Permission Level Distribution:')
df_silver.groupBy('PERMISSION_LEVEL').count().orderBy(F.col('count').desc()).show()

print('📊 Risk Level Distribution:')
df_silver.groupBy('RISK_LEVEL').count().orderBy(F.col('count').desc()).show()

print('📊 Account Category Distribution:')
df_silver.groupBy('ACCOUNT_CATEGORY').count().orderBy(F.col('count').desc()).show()

print('=' * 55)
print('  SECTION 5 COMPLETE: Quality Summary Printed')
print('  Review distributions above before proceeding.')
print('=' * 55)

## Section 6 — Admin Account Detection
Flags admin accounts **before** GenAI enrichment so they receive a standard note rather than a false-positive risk alert.

**Why is this step necessary?**  
Admin accounts legitimately hold Full Control across catalog types as part of their operational role. Without flagging, the GenAI risk narrative agent would produce high-risk alerts for expected behaviour — reducing trust in the Risk Dashboard.

**Two detection mechanisms (either fires the flag):**
1. **Name-based** — case-insensitive substring match against `ACCOUNT_NAME` using `ADMIN_NAME_PATTERNS`
2. **Breadth-based** — any account with Full Control on `ADMIN_FC_THRESHOLD`+ distinct catalog types is flagged regardless of name

**Output column:** `IS_ADMIN_ACCOUNT` (boolean)
- `True` → receives `ADMIN_NARRATIVE` constant, skips GenAI
- `False` → processed by Agent B in Section 8

In [ ]:
# Name-based condition — OR across all configured patterns
admin_name_condition = F.lit(False)
for pattern in ADMIN_NAME_PATTERNS:
    admin_name_condition = admin_name_condition | F.lower(F.col('ACCOUNT_NAME')).contains(pattern.lower())

# Breadth-based — Full Control on ADMIN_FC_THRESHOLD+ catalog types
admin_breadth_df = (
    df_silver
    .filter(F.col('PERMISSION_LEVEL') == 'Full Control')
    .groupBy('ACCOUNT_NAME')
    .agg(F.countDistinct('CATALOG_TYPE').alias('FC_TYPE_COUNT'))
    .filter(F.col('FC_TYPE_COUNT') >= ADMIN_FC_THRESHOLD)
    .select('ACCOUNT_NAME', F.lit(True).alias('IS_BREADTH_ADMIN'))
)

df_silver = df_silver.join(admin_breadth_df, on='ACCOUNT_NAME', how='left')
df_silver = df_silver.withColumn(
    'IS_ADMIN_ACCOUNT',
    admin_name_condition | (F.col('IS_BREADTH_ADMIN') == True)
).drop('IS_BREADTH_ADMIN')

admin_count = df_silver.filter(F.col('IS_ADMIN_ACCOUNT') == True).select('ACCOUNT_NAME').distinct().count()
total_count = df_silver.select('ACCOUNT_NAME').distinct().count()

print('=' * 55)
print('  SECTION 6 COMPLETE: Admin Account Detection')
print(f'  Total distinct accounts  : {total_count:,}')
print(f'  Admin accounts flagged   : {admin_count:,}')
print(f'  Non-admin (GenAI input)  : {total_count - admin_count:,}')
print('=' * 55)

## Section 7 — OCI GenAI Client
Initialises the OCI Generative AI Inference client using **Resource Principal** authentication — the correct auth method for AIDP notebooks. No API key or config file required.

**Model:** `google.gemini-2.5-pro` — available in this AIDP instance's Default Catalog AI Models.  
Uses `GenericChatRequest` — required for Gemini, Grok, and Llama models. Switch to `CohereChatRequest` for Cohere models.

**Error handling:** `call_genai()` returns a `[GENAI_ERROR]` prefixed string on failure rather than raising. Callers check for this prefix to detect failed generations without interrupting the enrichment loop.

> **Alternative — AIDP SQL approach:** If you prefer to call GenAI via Spark SQL using AIDP's built-in model function:
> ```sql
> SELECT account_name, ai_generate(prompt_col) AS risk_narrative FROM prompt_temp_view
> ```
> Replace `ai_generate` with the registered function name visible under Default Catalog → OCI AI Models.

In [ ]:
import oci
from oci.generative_ai_inference.models import (
    ChatDetails, OnDemandServingMode, GenericChatRequest, UserMessage, TextContent
)


def get_genai_client():
    # Resource Principal — correct auth for AIDP notebooks
    signer = oci.auth.signers.get_resource_principals_signer()
    return oci.generative_ai_inference.GenerativeAiInferenceClient(
        config={}, signer=signer, service_endpoint=OCI_GENAI_ENDPOINT
    )


def call_genai(client, prompt, max_tokens=GENAI_MAX_TOKENS, temperature=GENAI_TEMPERATURE):
    """Call OCI GenAI chat endpoint. Returns generated text or [GENAI_ERROR] on failure."""
    try:
        text_content      = TextContent()
        text_content.text = prompt
        message           = UserMessage()
        message.content   = [text_content]
        chat_request              = GenericChatRequest()
        chat_request.messages     = [message]
        chat_request.max_tokens   = max_tokens
        chat_request.temperature  = temperature
        chat_detail                = ChatDetails()
        chat_detail.serving_mode   = OnDemandServingMode(model_id=OCI_GENAI_MODEL_ID)
        chat_detail.chat_request   = chat_request
        chat_detail.compartment_id = OCI_GENAI_COMPARTMENT_ID
        response = client.chat(chat_detail)
        return response.data.chat_response.choices[0].message.content[0].text.strip()
    except Exception as e:
        return f'[GENAI_ERROR] {str(e)[:200]}'


print('=' * 55)
print('  SECTION 7 COMPLETE: OCI GenAI Client Ready')
print(f'  Model : {OCI_GENAI_MODEL_ID}')
print(f'  Auth  : Resource Principal')
print('=' * 55)

## Section 8 — Agent B: Risk Narrative per Account
Generates a plain-English 2–3 sentence risk summary for each distinct `ACCOUNT_NAME` using OCI GenAI.

**Why driver-side collection?**  
ACL data is small (typically <500 distinct accounts). Collecting to the driver and calling GenAI sequentially is simpler and easier to debug than Spark executor → external API call patterns, which require careful connection pool and rate limit management across distributed nodes.

**Admin accounts** (flagged in Section 6) are excluded from GenAI calls and assigned `ADMIN_NARRATIVE` directly — no false-positive risk alerts for expected admin behaviour.

**Output column:** `RISK_NARRATIVE`
- Admin accounts → `ADMIN_NARRATIVE` constant
- Non-admin accounts → GenAI-generated 2–3 sentence summary
- Accounts with no profile data → `"No profile available"`
- Failed calls → `[GENAI_ERROR]` prefixed message

The narrative is joined back on `ACCOUNT_NAME` — every row for a given account receives the same narrative.

In [ ]:
def build_risk_narrative_prompt(account_name, account_category, profile):
    profile_lines = '\n'.join([
        f"  - {p['catalog_type_label']}: {p['permission_level']} ({p['item_count']} object(s), risk: {p['risk_level']})"
        for p in profile
    ])
    return f"""You are an Oracle Analytics Cloud security analyst reviewing access control lists.

Write a 2-3 sentence plain-English risk assessment for the {account_category.lower()} account '{account_name}'. Be specific and factual based only on the permission profile below. Do not use bullet points. Do not recommend generic best practices unless there is a specific reason based on this profile.

Permission profile:
{profile_lines}

Risk assessment:"""


print('\n[AGENT B] Building risk narratives via OCI GenAI...')

genai_client = get_genai_client()

# Separate admin and non-admin accounts
admin_accounts = set(
    row.ACCOUNT_NAME
    for row in df_silver.filter(F.col('IS_ADMIN_ACCOUNT') == True)
                        .select('ACCOUNT_NAME').distinct().collect()
    if row.ACCOUNT_NAME
)

# Build permission profiles for non-admin accounts
profile_rows = (
    df_silver.filter(F.col('IS_ADMIN_ACCOUNT') == False)
             .filter(F.col('ACCOUNT_NAME').isNotNull())
             .groupBy('ACCOUNT_NAME', 'ACCOUNT_CATEGORY', 'CATALOG_TYPE_LABEL', 'PERMISSION_LEVEL', 'RISK_LEVEL')
             .agg(F.count('*').alias('item_count'))
             .collect()
)

account_profiles = defaultdict(lambda: {'category': 'Unknown', 'permissions': []})
for row in profile_rows:
    account_profiles[row.ACCOUNT_NAME]['category'] = row.ACCOUNT_CATEGORY
    account_profiles[row.ACCOUNT_NAME]['permissions'].append({
        'catalog_type_label': row.CATALOG_TYPE_LABEL,
        'permission_level':   row.PERMISSION_LEVEL,
        'risk_level':         row.RISK_LEVEL,
        'item_count':         row.item_count,
    })

narrative_lookup = {}

# Admin accounts — standard override, no GenAI call
for acct in admin_accounts:
    narrative_lookup[acct] = ADMIN_NARRATIVE

print(f'  Admin accounts  : {len(admin_accounts)} — standard narrative applied')
print(f'  Non-admin       : {len(account_profiles)} — calling OCI GenAI...')

for i, (account_name, profile) in enumerate(account_profiles.items()):
    prompt    = build_risk_narrative_prompt(account_name, profile['category'], profile['permissions'])
    narrative = call_genai(genai_client, prompt)
    narrative_lookup[account_name] = narrative
    if (i + 1) % 10 == 0 or (i + 1) == len(account_profiles):
        print(f'  Progress: {i + 1}/{len(account_profiles)} narratives generated')
    time.sleep(GENAI_RATE_LIMIT_SLEEP)

narrative_df = spark.createDataFrame(
    [(name, text) for name, text in narrative_lookup.items()],
    schema=['ACCOUNT_NAME', 'RISK_NARRATIVE']
)

df_silver = df_silver.join(narrative_df, on='ACCOUNT_NAME', how='left')
df_silver = df_silver.fillna({'RISK_NARRATIVE': 'No profile available'})

print('=' * 55)
print('  SECTION 8 COMPLETE: Agent B — Risk Narratives Done')
print(f'  Admin (override) : {len(admin_accounts)}')
print(f'  GenAI generated  : {len(account_profiles)}')
print('=' * 55)

## Section 9 — Agent C: Owner Inference for Orphaned Objects
For catalog objects where `ITEM_OWNER` is null, empty, or matches a known service/shared account pattern, calls OCI GenAI to suggest the most likely responsible owner.

**Why is owner inference useful?**  
Objects without a clear human owner are governance blind spots — permissions may never be reviewed, stale access accumulates, and sensitive data may be more widely accessible than intended. Inference gives admins a starting point for accountability conversations.

**Trigger conditions:**
1. `ITEM_OWNER` is null or empty
2. `ITEM_OWNER` matches any `SERVICE_ACCOUNT_PATTERNS`

**Prompt context:** object name, catalog type, decoded path, and up to 10 account names with access. The account list often hints at likely ownership — a dataset accessed only by one team is likely owned by someone from that team.

**Output columns:**
- `INFERRED_OWNER` — suggested owner, `"Unable to infer"`, or `"N/A — Owner on record"`
- `INFERRED_OWNER_NOTE` — one-sentence rationale from the model

In [ ]:
def build_owner_inference_prompt(item_name, item_path, catalog_type_label, subjects):
    subjects_str = ', '.join(subjects[:10]) if subjects else 'none on record'
    return f"""You are an Oracle Analytics Cloud governance analyst.

The following catalog object has no identifiable human owner. Based only on the information provided, suggest the most likely responsible owner in one sentence and explain your reasoning. If you cannot infer an owner, say 'Unable to infer' and briefly explain why.

Object name:          {item_name}
Object type:          {catalog_type_label}
Object path:          {item_path}
Accounts with access: {subjects_str}

Suggested owner and rationale (one sentence):"""


print('\n[AGENT C] Identifying orphaned objects for owner inference...')

orphan_df = (
    df_silver
    .filter(
        F.col('ITEM_OWNER').isNull()
        | (F.col('ITEM_OWNER') == '')
        | F.col('ITEM_OWNER').rlike('(?i)(' + '|'.join(SERVICE_ACCOUNT_PATTERNS) + ')')
    )
    .select('ITEM_ID', 'ITEM_NAME', 'ITEM_PATH', 'CATALOG_TYPE_LABEL', 'ACCOUNT_NAME', 'ITEM_OWNER')
    .distinct()
)

orphan_count = orphan_df.select('ITEM_ID').distinct().count()
print(f'  Orphaned/service-owned objects: {orphan_count}')

orphan_rows     = orphan_df.collect()
object_subjects = defaultdict(list)
object_meta     = {}

for row in orphan_rows:
    iid = row.ITEM_ID
    if row.ACCOUNT_NAME:
        object_subjects[iid].append(row.ACCOUNT_NAME)
    object_meta[iid] = {
        'item_name':          row.ITEM_NAME  or '',
        'item_path':          row.ITEM_PATH  or '',
        'catalog_type_label': row.CATALOG_TYPE_LABEL or '',
    }

inference_lookup = {}
print(f'  Calling OCI GenAI for {len(object_meta)} object(s)...')

for i, (item_id, meta) in enumerate(object_meta.items()):
    prompt   = build_owner_inference_prompt(
        meta['item_name'], meta['item_path'], meta['catalog_type_label'],
        object_subjects.get(item_id, [])
    )
    response = call_genai(genai_client, prompt, max_tokens=150)
    if response.lower().startswith('unable to infer'):
        inference_lookup[item_id] = ('Unable to infer', response)
    elif response.startswith('[GENAI_ERROR]'):
        inference_lookup[item_id] = ('Error', response)
    else:
        inference_lookup[item_id] = ('See note', response)
    if (i + 1) % 10 == 0 or (i + 1) == len(object_meta):
        print(f'  Progress: {i + 1}/{len(object_meta)} inferences generated')
    time.sleep(GENAI_RATE_LIMIT_SLEEP)

inference_df = spark.createDataFrame(
    [(iid, owner, note) for iid, (owner, note) in inference_lookup.items()],
    schema=['ITEM_ID', 'INFERRED_OWNER', 'INFERRED_OWNER_NOTE']
)

df_silver = df_silver.join(inference_df, on='ITEM_ID', how='left')
df_silver = df_silver.fillna({'INFERRED_OWNER': 'N/A — Owner on record', 'INFERRED_OWNER_NOTE': ''})

print('=' * 55)
print('  SECTION 9 COMPLETE: Agent C — Owner Inference Done')
print(f'  Objects with inferred owners: {len(inference_lookup)}')
print(f'  Total Silver columns        : {len(df_silver.columns)}')
print('=' * 55)

## Section 10 — Write Stage 1 (Standard Catalog Silver / Delta)
Writes the enriched Silver DataFrame to the AIDP Standard Catalog as a managed Delta table.

Same pattern as Bronze Stage 1 — Delta format, schema evolution, ACID transactions, portable across AIDP instances.

The Silver Delta table is the preferred downstream source for the Gold notebook and for any OAC dataset connection that requires the full enriched column set (`PERMISSION_LEVEL`, `RISK_LEVEL`, `RISK_NARRATIVE`, `IS_ADMIN_ACCOUNT`, `INFERRED_OWNER`, etc.).

In [ ]:
print(f'\n[WRITE] Stage 1 — Standard Catalog Silver (Delta)')
print(f'        Target: {SILVER_STAGE1}')

(df_silver.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(SILVER_STAGE1)
)

df_check_s1 = spark.table(SILVER_STAGE1)
print('=' * 55)
print('  SECTION 10 COMPLETE: Stage 1 Write Done')
print(f'  Table   : {SILVER_STAGE1}')
print(f'  Rows    : {df_check_s1.count():,}')
print(f'  Columns : {len(df_check_s1.columns)}')
print(f'  Format  : Delta')
print('=' * 55)

## Section 11 — Write Stage 2 (ADW via External Catalog)
Reads from the Stage 1 Silver Delta table and writes to ADW via the AIDP External Catalog. Same constraints as Bronze Stage 2.

**Why Stage 2 is still needed:**  
The ADW Silver table powers the existing OAC report workbook. Until OAC canvases are repointed to the Standard Catalog Silver connection, Stage 2 is required for report continuity.

**External Catalog constraints:**
- No Delta format — ADW does not support it
- No `CREATE SCHEMA` — 502 from AIDP Metastore
- 3-part path required
- Full overwrite — Silver is always fully regenerated from Bronze

In [ ]:
print(f'\n[WRITE] Stage 2 — ADW via External Catalog Silver')
print(f'        Source: {SILVER_STAGE1}  (Delta)')
print(f'        Target: {SILVER_STAGE2}')

df_from_delta = spark.table(SILVER_STAGE1)

(df_from_delta.write
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(SILVER_STAGE2)
)

df_check_s2 = spark.table(SILVER_STAGE2)
print('=' * 55)
print('  SECTION 11 COMPLETE: Stage 2 Write Done')
print(f'  Table   : {SILVER_STAGE2}')
print(f'  Rows    : {df_check_s2.count():,}')
print(f'  Columns : {len(df_check_s2.columns)}')
print(f'  Status  : Queryable in AIDP Master Catalog + OAC')
print('=' * 55)

print('\n' + '=' * 60)
print('  🏁 SILVER PIPELINE COMPLETE')
print('=' * 60)